# Exp 2 · Minimal connectivity-only GCN

Config `configs/final/experiments/exp2_conn_minimal.yaml` (10 variants; PLV and |ciPLV| are separate results, never merged):

- `plv__delta`
- `plv__theta`
- `plv__alpha`
- `plv__beta`
- `plv__gamma`
- `ciplv__delta`
- `ciplv__theta`
- `ciplv__alpha`
- `ciplv__beta`
- `ciplv__gamma`

**Protocol:** frozen participant split of ds004504 (52 development / 13 holdout). This notebook runs `xu_dev_cv`
only: 5-fold participant-grouped CV inside the development set. The holdout (`xu_holdout`: fit on all 52
development participants, evaluate once on the 13 holdout participants) is NOT run here: it is evaluated once, for
the frozen final analysis only, in notebook 99. Fixed hyperparameters (protocol `gnn_fixed60_v1`): exactly 60 epochs, Adam lr 1e-3, weight decay 1e-4, batch 32, participant- and class-balanced BCE, no early stopping, no search; node features z-scored with training statistics; score P(AD) (threshold 0.5). Participant score = mean window score; participant-level metrics
(pooled out-of-fold ROC-AUC, mean fold AUC, balanced accuracy, sensitivity, specificity, confusion matrix, DeLong /
bootstrap / Wilson CIs) are primary. The store's participant QC must have no unresolved category A finding.
Atlas: the analysis atlas recorded in notebook 04 (`--atlas selected`).
Graph construction: the recorded development-only decision of notebook 03 (`--graph selected`).

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

**Contexts:** LCMV and wMNE, window mode B (all accepted windows, primary) then A (10 windows per participant, sensitivity).
Edit `CONTEXTS` below only to run a subset; the order is the recommended order.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.
 Use a **GPU runtime** (T4).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh as _sh

def sh(cmd, cwd=None):  # every script runs with the file backend: Colab's inline backend breaks figures in scripts
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

## 1. Contexts to run (the only cell to edit, if at all)

In [ ]:
EXP = 'exp2'
CONTEXTS = [('lcmv', 'B'), ('wmne', 'B'), ('lcmv', 'A'), ('wmne', 'A')]     # (inverse, window mode), in this order
GRAPH = 'selected'
ATLAS = 'selected'      # the analysis atlas recorded in notebook 04 (FINAL_v1/decisions/atlas.json)
import torch
print('runtime:', ('GPU ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU only')

## 2. Status, then run (resumable; one experiment only)

In [ ]:
ERRORS = {}
g = f" --graph {GRAPH}" if GRAPH else ""
for inverse, mode in CONTEXTS:
    inv = 'lcmv' if inverse == 'sensor' else inverse
    print(f'==== {EXP} | {inverse} | mode {mode}')
    sh(f"python scripts/final.py status --exp {EXP} --inverse {inv} --mode {mode} --atlas {ATLAS}{g}")
    try:
        sh(f"python scripts/final.py run --exp {EXP} --inverse {inv} --mode {mode} --atlas {ATLAS}{g}")
    except RuntimeError as exc:
        ERRORS[(inverse, mode)] = exc
        print(f'{inverse} / mode {mode}: some units failed; details after the results')

## 3. Results and diagnostics (saved outputs only; diagnostic figures are not paper figures)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
for inverse, mode in CONTEXTS:
    inv = 'lcmv' if inverse == 'sensor' else inverse
    sh(f"python scripts/final.py report --exp {EXP} --inverse {inv} --mode {mode} --atlas {ATLAS}{g}")
    atlas = 'dk68' if inverse == 'sensor' else json.load(open(FINAL + '/decisions/atlas.json'))['atlas']
    rep = latest(f"{FINAL}/results/{inverse}/mode_{mode}/{atlas}/*/*/_reports/report_*")
    if rep:
        print('====', inverse, 'mode', mode, rep)
        t = pd.read_csv(rep + '/results.csv')
        cols = ['variant', 'setting', 'auc', 'auc_ci_low', 'auc_ci_high', 'mean_fold_auc', 'min_fold_auc',
                'max_fold_auc', 'balanced_accuracy', 'sensitivity', 'specificity', 'tn', 'fp', 'fn', 'tp']
        display(t[[c for c in cols if c in t]].round(3))
        for png in sorted(glob.glob(rep + '/*.png')):
            display(Image(filename=png))
if ERRORS:
    raise RuntimeError(f'units did not complete in {sorted(ERRORS)}; completed units are saved. Run all again.')